# ClipForge — Colab GPU

対象アカウント: **n.yu17919191@gmail.com**。右上のアカウントを確認し、ランタイムのタイプで **T4 GPU** を選択してください。GPU割り当てがない場合は停止します。

上からセットアップ・Runnerのセルを実行します。Runnerは**非公開 ai-model-workers** の1ジョブだけを受け取ります。公式の利用条件・計算ユニット残高を確認してください。

In [ ]:
import os, subprocess
from pathlib import Path
SOURCE_SHA = "496c76a87457bf81544c3227038fa4b1f42fe2fb"
REPOSITORY = "https://github.com/nyu1791-collab/ClipForge.git"
ROOT = Path("/content/ClipForge")
subprocess.run(["nvidia-smi"], check=True)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", SOURCE_SHA], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
    head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
    if origin != REPOSITORY or head != SOURCE_SHA or dirty:
        raise RuntimeError("既存 /content/ClipForge が異なります。変更を保存し、新しいランタイムで実行してください。")
subprocess.run(["bash", "tools/colab_setup.sh"], cwd=ROOT, check=True)
print("GPUセットアップ完了。コードSHA:", SOURCE_SHA)


## 非公開GitHub Actionsへ接続

非公開workerの登録トークンをColab Secretsの `CLIPFORGE_RUNNER_TOKEN` に設定するか、実行時の非表示入力で渡します。GitHub PATを保存しないでください。

出力が `Listening for Jobs`、GitHubで `colab-gpu` ラベルのランナーが `Idle` になって接続完了です。このセルは待機中のままにしてください。

In [ ]:
import getpass, os, subprocess
try:
    from google.colab import userdata
    registration_token = userdata.get("CLIPFORGE_RUNNER_TOKEN")
except Exception:
    registration_token = getpass.getpass("非公開 ai-model-workers のランナー登録トークン: ")
if not registration_token:
    raise RuntimeError("登録トークンが必要です。PATではなくRunner registration tokenを指定してください。")
runner_env = os.environ.copy()
runner_env["RUNNER_TOKEN"] = registration_token
try:
    subprocess.run(["bash", "tools/colab_runner.sh"], cwd=ROOT, env=runner_env, check=True)
finally:
    runner_env.pop("RUNNER_TOKEN", None)
    registration_token = None
print("Runner終了。1ジョブで登録解除されます。次のジョブは新しい登録トークンで起動してください。")


## 任意: Colab内で直接処理

Runnerと別の実行方式です。通常は無効のままにします。有効にする場合はモデルIDを明示し、Secretsに `GEMINI_API_KEY` を設定します。候補動画・音声・ASRを指定モデルへ送信します。Driveへのコピー時間は発生します。

In [ ]:
# Self-hosted Runnerを使わず、Colab内で処理する場合だけTrueにします。
DIRECT_RUN = False
INPUT_FILE = "/content/drive/MyDrive/source.mp4"
INPUT_URL = ""  # 直接動画ファイルのHTTPS URL。署名URLを保存セルへ残さないこと。
GEMINI_MODEL = ""  # 利用する動画対応モデルのAPI識別子を明示
if DIRECT_RUN:
    import os, subprocess
    from datetime import datetime, timezone
    from google.colab import drive, userdata
    if not GEMINI_MODEL:
        raise RuntimeError("GeminiモデルIDを指定してください。自動モデル選択は行いません。")
    drive.mount("/content/drive")
    job = datetime.now(timezone.utc).strftime("job-%Y%m%d-%H%M%S")
    environment = os.environ.copy()
    environment["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    arguments = ["--url", INPUT_URL] if INPUT_URL else ["--input", INPUT_FILE]
    arguments += ["--gemini-model", GEMINI_MODEL, "--config", "config.example.json",
                  "--output", "/content/clipforge-output/" + job,
                  "--drive-output", "/content/drive/MyDrive/ClipForge_Output/" + job]
    try:
        subprocess.run(["bash", "-c", 'source tools/colab_cuda_env.sh && exec .venv/bin/python tools/colab_run.py "$@"',
                        "clipforge-colab", *arguments], cwd=ROOT, env=environment, check=True)
    finally:
        environment.pop("GEMINI_API_KEY", None)
